# Worked Example: PSD and FOOOF Reward Contrast

## Goal
Compare reward vs no-reward spectra on feedback epochs. FOOOF uses `analysis_utils` — see {doc}`11_advanced_utility_interoperability`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from LFPAnalysis import load_lfp
from LFPAnalysis.config import LoadConfig

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = load_lfp(LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne'))
epochs.metadata = beh[['reward', 'rpe']]
chan = 'racas1-racas2'
reward_psd = epochs['reward == 1'].copy().pick([chan]).compute_psd(fmin=1, fmax=80, verbose=False)
loss_psd = epochs['reward == 0'].copy().pick([chan]).compute_psd(fmin=1, fmax=80, verbose=False)

## Plot PSD contrast

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(reward_psd.freqs, reward_psd.get_data()[0, 0], label='reward')
ax.semilogy(loss_psd.freqs, loss_psd.get_data()[0, 0], label='no reward')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD', title=f'{chan} feedback-locked')
ax.legend()
fig.tight_layout()
plt.show()

## FOOOF via advanced utility (subset for speed)

In [ ]:
from LFPAnalysis import analysis_utils

epochs_sub = epochs.copy().pick([chan])[:10]
fooof_kwargs = {
    'peak_width_limits': (1, 12),
    'min_peak_height': 0.0,
    'peak_threshold': 2.0,
    'max_n_peaks': 6,
    'freq_range': (1, 40),
}
_, fooof_table = analysis_utils.FOOOF_compute_epochs(
    epochs_sub, tmin=float(epochs_sub.times[0]), tmax=float(epochs_sub.times[-1]), **fooof_kwargs
)
print(fooof_table.head())

## Next step

Advanced utility interoperability: 11_advanced_utility_interoperability. Next chapter: time-frequency.